# Anthropic Skilljar — Playground
Notebook latihan interaktif untuk course **Anthropic Skilljar: Building with the Claude API**.  
Jalankan cell secara berurutan. Pastikan file `.env` berisi `ANTHROPIC_API_KEY`.

---
## ⚙️ Setup

In [17]:
%pip install anthropic python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from dotenv import load_dotenv
from anthropic import Anthropic
from pprint import pprint

In [19]:
load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

print("Client ready:", model)

Client ready: claude-sonnet-4-5


---
## C. Making a Request

In [20]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)

print(message.content)

[TextBlock(citations=None, text='Quantum computing is a type of computation that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.', type='text')]


In [21]:
# Inspect the full response object
print("Full response object:", message)
print("Stop reason :", message.stop_reason)
print("Input tokens :", message.usage.input_tokens)
print("Output tokens:", message.usage.output_tokens)

Full response object: Message(id='msg_bdrk_01RSMZRbKZqsG8cm8Tz4j8eT', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computation that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.', type='text')], model='claude-sonnet-4-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=17, output_tokens=40, output_tokens_details=None, server_tool_use=None, service_tier=None))
Stop reason : end_turn
Input tokens : 17
Output tokens: 40


---
## D. Multi-Turn Conversations

In [22]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        # "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

print("Helpers defined.")

Helpers defined.


In [23]:
messages = []

add_user_message(messages, "Define quantum computing in one sentence.")
answer = chat(messages)
print("Turn 1:", answer)

add_assistant_message(messages, answer)
add_user_message(messages, "Write one more sentence expanding on that.")
answer2 = chat(messages)
print("Turn 2:", answer2)

Turn 1: Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain complex problems exponentially faster than classical computers.
Turn 2: Unlike classical computers that use bits representing either 0 or 1, quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, allowing them to explore many possible solutions to a problem at once.


---
## E. System Prompts

In [24]:
# Without system prompt — Claude gives the full answer immediately
messages = []
add_user_message(messages, "How do I solve 5x + 2 = 3 for x?")
print("No system prompt:")
print(chat(messages))

No system prompt:
# Solving 5x + 2 = 3

To solve for x, isolate it by using inverse operations:

**Step 1:** Subtract 2 from both sides
- 5x + 2 - 2 = 3 - 2
- 5x = 1

**Step 2:** Divide both sides by 5
- 5x/5 = 1/5
- x = 1/5

**Answer: x = 1/5** (or 0.2 as a decimal)

You can check this by substituting back: 5(1/5) + 2 = 1 + 2 = 3 ✓


In [25]:
# With system prompt — acts like a Socratic math tutor
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to the solution step by step using hints.
"""

messages = []
add_user_message(messages, "How do I solve 5x + 2 = 3 for x?")
print("With math tutor system prompt:")
print(chat(messages, system=system))

With math tutor system prompt:
Great question! Let's work through this together step by step.

First, think about what we're trying to do here. We want to get **x by itself** on one side of the equation.

Right now we have: 5x + 2 = 3

**Hint 1:** What's "attached" to the x term (5x) that we need to deal with first?

**Hint 2:** What operation could we perform on both sides of the equation to remove that +2?

Take a try at the first step, and let me know what you get! 🎯


---
## F. Temperature

In [26]:
# Low temperature — deterministic, consistent
messages = []
add_user_message(messages, "Give me a one-sentence movie plot idea.")

print("temperature=0.0")
for _ in range(15):
    print(" -", chat(messages, temperature=0.0))

temperature=0.0
 - A retired astronaut receives a mysterious signal from deep space that matches the voice pattern of her daughter who disappeared on a Mars colony twenty years ago.
 - A time traveler keeps accidentally preventing their own parents from meeting, but each fix creates a worse timeline where they're related to increasingly bizarre historical figures.
 - A retiring astronaut discovers their entire space career was a simulation, but the skills they learned turn out to be the only way to stop a real alien invasion.
 - A grief-stricken astronaut discovers that the alien signal she's been decoding for years is actually a message from her future self, warning her not to make first contact.
 - A time-traveling food critic accidentally prevents the invention of pizza and must race through history to fix the timeline before Italian cuisine—and their career—disappears forever.
 - A lonely astronaut discovers that the AI running her ship has been hiding the fact that Earth was destr

In [27]:
# High temperature — creative, varied
print("temperature=1.0")
for _ in range(15):
    print(" -", chat(messages, temperature=1.0))

temperature=1.0
 - A time-traveling historian accidentally prevents the invention of music and must race to restore the timeline before silence erases humanity's ability to feel emotion.
 - A retired astronaut discovers their old space station has drifted back into orbit with something alive inside it.
 - A time-traveling historian accidentally prevents their own birth while trying to document their grandparents' first meeting, and must race against their fading existence to fix the timeline.
 - A reclusive lighthouse keeper discovers their lighthouse beam is actually preventing an ancient sea creature from returning home, forcing them to choose between protecting ships or reuniting a family.
 - A time-traveling historian accidentally prevents the invention of language and must communicate through interpretive dance to fix the timeline before humanity loses the ability to speak forever.
 - A time-traveling librarian accidentally prevents the invention of written language and must convi

---
## G. Response Streaming

In [28]:
messages = []
add_user_message(messages, "Write a 3-sentence description of a fictional database product.")

print("Streaming output:")
with client.messages.stream(
    model=model,
    max_tokens=300,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print("\n-" + text, end="", flush=True)

    final = stream.get_final_message()

print("\n\nFinal stop reason:", final.stop_reason)
print("Total output tokens:", final.usage.output_tokens)

Streaming output:

-**
-Data
-V
-ault Pro
-**
- is
- an
- intelligent
- cloud
--native
- database platform that combines
- the speed
- of
- in
--memory processing
- with advanced
- AI
--
-powered query
- optimization. It features automatic
- schema evolution, real-time data re
-plication across
- global
- regions
-, and built
--in compliance tools
- for
- GDPR and
- HIPAA requirements
-.
- Designed
- for
- modern enterprises
-,
- Data
-Vault Pro scales
- seam
-lessly from startup
- to Fortune
- 500 work
-loads while maintaining 
-99.99
-% uptime.

Final stop reason: end_turn
Total output tokens: 99


---
## H. Structured Data — Assistant Prefilling + Stop Sequences

In [29]:
import json

messages = []
add_user_message(messages, "Generate a very short AWS EventBridge rule as JSON.")
add_assistant_message(messages, "```json")

raw = chat(messages, stop_sequences=["```"])
print("Raw text from Claude:")
print(raw)

parsed = json.loads(raw.strip())
print("\nParsed JSON:")
print(json.dumps(parsed, indent=2))

Raw text from Claude:

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}


Parsed JSON:
{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "running"
    ]
  }
}


In [30]:
# Try it yourself — change the prompt to get a different JSON structure
messages = []
add_user_message(messages, "Generate a short JSON object representing a user profile with name, email, and role.")
add_assistant_message(messages, "```json")

raw = chat(messages, stop_sequences=["```"])
parsed = json.loads(raw.strip())
print(json.dumps(parsed, indent=2))

{
  "name": "John Smith",
  "email": "john.smith@example.com",
  "role": "admin"
}


---
## 🧪 Free Sandbox
Eksperimen bebas — ubah prompt, system, temperature, stop_sequences sesuka kamu.

In [31]:
messages = []
system   = ""  # optional
temp     = 1.0

add_user_message(messages, "Your prompt here")

print(chat(messages, system=system or None, temperature=temp))

I'm ready to help! However, I don't see a specific prompt or question in your message. Could you please provide the details of what you'd like assistance with?


# Structured data

In [ ]:
messages = []
add_user_message(messages,"Generate a very short event bridge rule as json")
add_assistant_message(messages,"```json")

text = chat(messages, stop_sequences=["```"])
print("Raw text from Claude:")
print(text)

add_user_message(messages,"Generate simple python code to find factorial of a number")
add_assistant_message(messages,"```python")
text = chat(messages, stop_sequences=["```"])
print("Raw text from Claude:")
print(text)

add_user_message(messages,"Generate sample data for users table with 5 users im csv format")
add_assistant_message(messages,"```csv")
text = chat(messages, stop_sequences=["```"])
print("Raw text from Claude:")
print(text)

Raw text from Claude:

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}

Raw text from Claude:

import json

# JSON data
rule_json = '''
{
  "Name": "MyRule",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"]
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}
'''

# Parse JSON and print rule name
rule = json.loads(rule_json)
print(rule['Name'])

Raw text from Claude:

id,first_name,last_name,email,age,city,country,created_at
1,John,Smith,john.smith@email.com,28,New York,USA,2023-01-15
2,Sarah,Johnson,sarah.j@email.com,34,London,UK,2023-02-20
3,Michael,Chen,m.chen@email.com,42,Toronto,Canada,2023-03-10
4,Emma,Rodriguez,emma.rodriguez@email.com,25,Madrid,Spain,2023-04-05
5,David,Kumar,david.k@email.com,31,Mumbai,India,2023-05-12



# Prompt Eval

In [35]:
def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.
    Example output:
    ```json
    [
        {
            "task": "DEscription of task
        },
        ... additional
    ]
    ```
    
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code
    Please generate 3 objects.
    
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    
    return json.loads(text)



In [39]:
dataset = generate_dataset()
print(dataset)
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': "Write a Python function that parses an ARN (Amazon Resource Name) string and returns a dictionary with its components: partition, service, region, account-id, and resource. For example, 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket' should return {'partition': 'aws', 'service': 's3', 'region': 'us-east-1', 'account_id': '123456789012', 'resource': 'bucket/my-bucket'}."}, {'task': "Create a JSON object that defines an IAM policy allowing read-only access to a specific S3 bucket named 'company-logs'. The policy should permit s3:GetObject and s3:ListBucket actions only for resources under that bucket."}, {'task': "Write a regex pattern that validates AWS EC2 instance IDs. The pattern should match strings that start with 'i-' followed by either 8 or 17 hexadecimal characters (e.g., 'i-1234abcd' or 'i-0123456789abcdef0')."}]


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output